# CLIP × ViLT — Cross-Attention Fusion for Fake News Detection
## Fakeddit Dataset · 2-Way Classification (Real vs Fake) · Kaggle T4/P100

---

### Architecture Overview

```
                    ┌──────────────────────────────────────────────┐
                    │              INPUT SAMPLE                    │
                    │   image  +  title text  +  metadata          │
                    └──────┬───────────────┬──────────────┬────────┘
                           │               │              │
               ┌───────────▼───┐   ┌───────▼──────┐  ┌───▼─────────────┐
               │  CLIP Vision  │   │  ViLT Encoder│  │  Metadata MLP   │
               │  ViT-L/14     │   │  vilt-b32-mlm│  │  num+cat → 64d  │
               │  patch tokens │   │  text tokens │  └───────┬─────────┘
               │ [B,256, 1024] │   │ [B, 40, 768] │          │
               └───────┬───────┘   └───────┬──────┘          │
                       │                   │                  │
               ┌───────▼───────────────────▼──────┐          │
               │   Bidirectional Cross-Attention   │          │
               │                                  │          │
               │  V_proj ──► CrossAttn(Q=V, KV=T) │          │
               │  T_proj ──► CrossAttn(Q=T, KV=V) │          │
               │      × FUSION_LAYERS (2)          │          │
               │                                  │          │
               │  MeanPool(V) ++ MeanPool(T)       │          │
               │        [B, 512]                   │          │
               └───────────────────┬──────────────┘          │
                                   │                          │
                          ┌────────▼──────────────────────────▼──┐
                          │        Concat  [B, 576]               │
                          │   LayerNorm → Linear → GELU → Drop    │
                          │   Linear → GELU → Drop → Linear(2)    │
                          └────────────────┬──────────────────────┘
                                           │
                                  ┌────────▼────────┐
                                  │  2-way Logits   │
                                  │  Real  /  Fake  │
                                  └─────────────────┘
```

### Model & Config Details

| Component | Detail |
|-----------|--------|
| **Vision backbone** | CLIP `openai/clip-vit-large-patch14` — ViT-L/14, patch tokens [B, 256, 1024] |
| **Text backbone** | ViLT `dandelin/vilt-b32-mlm` — word tokens [B, 40, 768] |
| **Fusion** | Bidirectional cross-attention (256-d, 8 heads, 2 layers) |
| **Metadata** | score · num_comments · upvote_ratio · domain · subreddit → 64-d MLP |
| **Freezing** | CLIP: last 2 vision blocks unfrozen · ViLT: last 2 encoder layers unfrozen |
| **Final head** | concat(512+64) → LayerNorm → 2-layer MLP → 2 logits |
| **Training** | 5-fold CV · 10 epochs · early-stop(patience=3) · label-smoothing=0.1 |

### Disabled Tasks
❌ 3-way classification  
❌ 6-way classification  
✅ 2-way only — 0 = Real, 1 = Fake


In [6]:
# ── Cell 1: Install Dependencies ─────────────────────────────────────────────
import sys, subprocess

pkgs = [
    "transformers>=4.40.0",   # CLIP + ViLT support
    "pillow",
    "tabulate",
    "scikit-learn",
]
for i, p in enumerate(pkgs, 1):
    print(f"[{i}/{len(pkgs)}] Installing {p} ...", end=" ", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
    print("✓")

import transformers, torch
print(f"\ntransformers {transformers.__version__}  |  torch {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM           : {mem:.1f} GB")


[1/4] Installing transformers>=4.40.0 ... ✓
[2/4] Installing pillow ... ✓
[3/4] Installing tabulate ... ✓
[4/4] Installing scikit-learn ... ✓

transformers 5.0.0  |  torch 2.10.0+cu128
CUDA available : True
GPU            : Tesla T4
VRAM           : 15.6 GB


In [7]:
# ── Cell 2: Imports & Configuration ──────────────────────────────────────────
import os, gc, json, random, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from PIL import Image
from transformers import (
    CLIPProcessor, CLIPModel,          # CLIP vision backbone
    ViltProcessor, ViltModel,          # ViLT text backbone
)
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from tabulate import tabulate

warnings.filterwarnings("ignore")

# ═══════════════════════════════ CONFIGURATION ════════════════════════════════
SEED          = 42
NUM_SAMPLES   = 12000       # larger sample reduces split noise and memorisation
NUM_FOLDS     = 10          # requested: more stable, lower-variance CV estimate
EPOCHS        = 10
BATCH_SIZE    = 8           # CLIP ViT-L/14 + ViLT on T4/P100
LEARNING_RATE = 8e-5        # fusion/head LR; backbone uses LR_BACKBONE below
LR_BACKBONE   = 1e-6
WEIGHT_DECAY  = 4e-2
DROPOUT_RATE  = 0.42
LABEL_SMOOTHING = 0.07
PATIENCE      = 4
NUM_CLASSES   = 2           # 2-way ONLY — 3-way & 6-way are DISABLED

# Leakage/overfitting controls. Metadata is fitted on train folds only.
# Subreddit/engagement metadata can produce fake 100% validation on Fakeddit,
# so keep only weak domain context by default.
USE_METADATA = True
USE_NUMERIC_METADATA = False
USE_DOMAIN_METADATA = True
USE_SUBREDDIT_METADATA = False
USE_GROUP_SPLIT = True
OVERFIT_GAP_PENALTY = 0.45
PERFECT_SCORE_WARNING = 0.995
UNFREEZE_LAST_BLOCKS = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Model identifiers ──────────────────────────────────────────────────────────
CLIP_MODEL = "openai/clip-vit-large-patch14"   # ViT-L/14  vision_hidden=1024
VILT_MODEL = "dandelin/vilt-b32-mlm"           # ViLT-B/32 hidden=768

# ── Backbone output dims ───────────────────────────────────────────────────────
CLIP_VISION_DIM = 1024    # ViT-L/14 hidden size
VILT_TEXT_DIM   = 768     # ViLT-B/32 hidden size

# ── Cross-attention fusion ────────────────────────────────────────────────────
FUSION_DIM    = 256       # projected common dim for Q/K/V
FUSION_HEADS  = 8         # multi-head attention heads
FUSION_LAYERS = 2         # stacked bidirectional cross-attn pairs

# ── Metadata ──────────────────────────────────────────────────────────────────
META_NUM_FEATURES    = 3     # score, num_comments, upvote_ratio
META_CAT_EMB_DIM     = 8
META_DOMAIN_VOCAB    = 100
META_SUBREDDIT_VOCAB = 1
META_HIDDEN          = 32

OUTPUT_DIR = "/kaggle/working"

# ── Seeds ─────────────────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device        : {DEVICE}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Vision model  : CLIP  ({CLIP_MODEL})  dim={CLIP_VISION_DIM}")
print(f"Text   model  : ViLT  ({VILT_MODEL})  dim={VILT_TEXT_DIM}")
print(f"Fusion        : Bidirectional cross-attention  "
      f"({FUSION_DIM}-d, {FUSION_HEADS} heads, {FUSION_LAYERS} layers)")
print(f"Task          : 2-way classification ONLY  (3-way & 6-way DISABLED)")
print(f"Config        : {EPOCHS} epochs | {NUM_FOLDS} folds | batch={BATCH_SIZE}")
print("="*70)


Device        : cuda
GPU           : Tesla T4
Vision model  : CLIP  (openai/clip-vit-large-patch14)  dim=1024
Text   model  : ViLT  (dandelin/vilt-b32-mlm)  dim=768
Fusion        : Bidirectional cross-attention  (256-d, 8 heads, 2 layers)
Task          : 2-way classification ONLY  (3-way & 6-way DISABLED)
Config        : 10 epochs | 10 folds | batch=8


In [8]:
# ── Cell 3: Fakeddit Dataset Loader (Kaggle) ──────────────────────────────────

def _is_fakeddit(path):
    """Quick header-only check for the required 2_way_label column."""
    try:
        sep = "\t" if path.suffix == ".tsv" else ","
        hdr = pd.read_csv(path, sep=sep, nrows=0)
        hdr.columns = hdr.columns.str.strip()
        return "2_way_label" in hdr.columns
    except Exception:
        return False


def load_fakeddit(n_samples=NUM_SAMPLES):
    """
    Scan /kaggle/input for a Fakeddit TSV/CSV with 2_way_label.
    Prefer train/validation metadata and avoid public test files for model
    selection, otherwise cross-validation reports an overly optimistic score.
    """
    print("Scanning /kaggle/input for Fakeddit dataset...\n")
    candidates = []
    for ext in ("*.tsv", "*.csv"):
        for p in Path("/kaggle/input").rglob(ext):
            name = p.name.lower()
            if not _is_fakeddit(p):
                print(f"  skip (no 2_way_label): {p.name}")
                continue
            if "test" in name:
                rank = 50
            elif "train" in name:
                rank = 0
            elif "valid" in name or "validate" in name:
                rank = 1
            else:
                rank = 10
            candidates.append((rank, len(str(p)), p))

    if not candidates:
        raise FileNotFoundError(
            "No Fakeddit file found in /kaggle/input!\n"
            "Add the Fakeddit dataset via Notebook Settings > Add Data.\n"
            "Required columns: 2_way_label, clean_title/title, id."
        )

    selected = sorted(candidates)[0][2]
    if "test" in selected.name.lower():
        print("WARNING: only a test/public file was found. Attach train/validate TSVs for reliable results.")
    print(f"  selected metadata: {selected}")
    return _load_file(selected, n_samples)


def _normalise_group_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()[:160]


def _load_file(path, n_samples):
    sep = "\t" if path.suffix == ".tsv" else ","
    df  = pd.read_csv(path, sep=sep, low_memory=False)
    df.columns = df.columns.str.strip()
    print(f"  Columns (first 15): {list(df.columns[:15])}")

    # Required columns
    text_col = next((c for c in ["clean_title","title","text"]
                     if c in df.columns), None)
    id_col   = "id" if "id" in df.columns else df.columns[0]
    if not text_col:
        raise ValueError(f"No text column found. Available: {list(df.columns[:10])}")

    # 2-way label ONLY — rename and filter
    df = df.rename(columns={id_col:"id", text_col:"text", "2_way_label":"label"})
    df["label"] = df["label"].astype(int)
    df = df[df["label"].isin([0, 1])]      # drop any non-binary rows

    # Metadata columns
    NUM_COLS = ["score", "num_comments", "upvote_ratio"]
    CAT_COLS = ["domain", "subreddit"]
    for c in NUM_COLS:
        df[c] = (pd.to_numeric(df[c], errors="coerce").fillna(0.0)
                 if c in df.columns else 0.0)
    for c in CAT_COLS:
        df[c] = (df[c].astype(str).str.strip().str.lower()
                 if c in df.columns else "unknown")
    print(f"  Numeric meta   : {[c for c in NUM_COLS if c in df.columns]}")
    print(f"  Categorical    : {[c for c in CAT_COLS if c in df.columns]}")

    # Text cleanup and duplicate grouping. Repeated/near-repeated titles across
    # random folds are a common source of inflated Fakeddit scores.
    df["text"] = (df["text"].astype(str).str.strip()
                             .str[:256]
                             .replace("", "[No text]"))
    df["group_key"] = df["text"].apply(_normalise_group_text)
    before_dupes = len(df)
    df = df.drop_duplicates(subset=["group_key", "label"]).reset_index(drop=True)
    print(f"  Duplicate title groups removed: {before_dupes - len(df)}")

    # Image paths
    img_dir = path.parent / "public_image_set"
    if not img_dir.exists():
        for d in path.parent.rglob("*"):
            if d.is_dir() and "image" in d.name.lower():
                img_dir = d; break
    if img_dir.exists():
        df["image_path"] = df["id"].apply(lambda x: _find_img(x, img_dir))
        before = len(df)
        df = df[df["image_path"].notna()]
        print(f"  Images matched : {len(df)}/{before}")
    else:
        print("  ⚠️  No image directory found — text+metadata only mode.")
        df["image_path"] = None

    return _balance(df, n_samples)


def _find_img(img_id, img_dir):
    for ext in (".jpg", ".jpeg", ".png", ".gif", ".webp"):
        p = img_dir / f"{img_id}{ext}"
        if p.exists():
            return str(p)
    return None


def _balance(df, n):
    if len(df) == 0:
        raise ValueError("No valid samples after filtering!")
    half = n // 2
    fake = df[df["label"]==1].sample(
        min(half, (df["label"]==1).sum()), random_state=SEED)
    real = df[df["label"]==0].sample(
        min(half, (df["label"]==0).sum()), random_state=SEED)
    k   = min(len(fake), len(real))
    out = (pd.concat([fake.iloc[:k], real.iloc[:k]])
             .sample(frac=1, random_state=SEED)
             .reset_index(drop=True))
    print(f"\n✓ Balanced: {len(out)} samples  "
          f"(Real={(out['label']==0).sum()} | Fake={(out['label']==1).sum()})")
    if "image_path" in out.columns:
        print(f"  With images: {out['image_path'].notna().sum()}")
    return out


print("Fakeddit loader ready ✓")


Fakeddit loader ready ✓


In [9]:
# ── Cell 4: Metadata Encoder ─────────────────────────────────────────────────
# Fit once on full balanced dataset; transform per-fold slice.
# Numeric   → log1p → z-score → float32 [N, 3]
# Categoric → top-K vocab    → int64   [N, 2]   (0 = <unk>)

class MetadataEncoderFitter:
    NUM_COLS    = ["score", "num_comments", "upvote_ratio"]
    CAT_COLS    = ["domain", "subreddit"]
    VOCAB_SIZES = {"domain": META_DOMAIN_VOCAB, "subreddit": META_SUBREDDIT_VOCAB}

    def __init__(self):
        self.cat_vocabs = {}
        self.num_mean   = {}
        self.num_std    = {}

    def fit(self, df):
        if not USE_METADATA:
            print("Metadata disabled: using zero metadata features to avoid leakage.")
            for col in self.NUM_COLS:
                self.num_mean[col] = 0.0
                self.num_std[col] = 1.0
            for col in self.CAT_COLS:
                self.cat_vocabs[col] = {}
            return self

        for col in self.NUM_COLS:
            if USE_NUMERIC_METADATA:
                v = np.log1p(np.abs(df[col].values.astype(float)))
                self.num_mean[col] = v.mean()
                self.num_std[col]  = v.std() + 1e-8
            else:
                self.num_mean[col] = 0.0
                self.num_std[col] = 1.0
        for col in self.CAT_COLS:
            enabled = ((col == "domain" and USE_DOMAIN_METADATA) or
                       (col == "subreddit" and USE_SUBREDDIT_METADATA))
            if enabled:
                top = df[col].value_counts().head(self.VOCAB_SIZES[col]).index.tolist()
                self.cat_vocabs[col] = {val: i+1 for i, val in enumerate(top)}
            else:
                self.cat_vocabs[col] = {}
            print(f"  [{col}] vocab size = {len(self.cat_vocabs[col])+1}  (incl. <unk>)")
        print("Metadata encoder fitted ✓")
        return self

    def transform(self, df):
        if not USE_METADATA:
            return (
                np.zeros((len(df), len(self.NUM_COLS)), dtype=np.float32),
                np.zeros((len(df), len(self.CAT_COLS)), dtype=np.int64),
            )

        num_cols = []
        for col in self.NUM_COLS:
            if USE_NUMERIC_METADATA:
                v = np.log1p(np.abs(df[col].values.astype(float)))
                num_cols.append((v - self.num_mean[col]) / self.num_std[col])
            else:
                num_cols.append(np.zeros(len(df), dtype=np.float32))
        num_arr = np.stack(num_cols, axis=1).astype(np.float32)   # [N, 3]

        cat_cols = []
        for col in self.CAT_COLS:
            enabled = ((col == "domain" and USE_DOMAIN_METADATA) or
                       (col == "subreddit" and USE_SUBREDDIT_METADATA))
            if enabled:
                ids = np.array(
                    [self.cat_vocabs[col].get(v, 0) for v in df[col].tolist()],
                    dtype=np.int64)
            else:
                ids = np.zeros(len(df), dtype=np.int64)
            cat_cols.append(ids)
        cat_arr = np.stack(cat_cols, axis=1)                       # [N, 2]
        return num_arr, cat_arr

    @property
    def domain_vocab_size(self):
        return (META_DOMAIN_VOCAB + 1) if USE_DOMAIN_METADATA else 1

    @property
    def subreddit_vocab_size(self):
        return (META_SUBREDDIT_VOCAB + 1) if USE_SUBREDDIT_METADATA else 1


print("MetadataEncoderFitter defined ✓")


MetadataEncoderFitter defined ✓


In [10]:
# ── Cell 5: Dual-Processor Dataset ───────────────────────────────────────────
#
# Two processors per sample:
#   CLIPProcessor  → clip_pixel_values  [3, 224, 224]   (ViT-L/14 expects 224px)
#   ViltProcessor  → input_ids, token_type_ids, attention_mask,
#                    vilt_pixel_values  [3, 384, 384]
#
# The ViLT processor also takes the image so ViLT can build its own
# patch embeddings (it is a joint VLP model, not a pure text encoder).

_CLIP_PLACEHOLDER  = Image.new("RGB", (224, 224), (127, 127, 127))
_VILT_PLACEHOLDER  = Image.new("RGB", (384, 384), (127, 127, 127))


class FakedditDualDataset(Dataset):
    def __init__(self, df, clip_proc, vilt_proc, meta_num, meta_cat):
        self.df        = df.reset_index(drop=True)
        self.clip_proc = clip_proc
        self.vilt_proc = vilt_proc
        self.meta_num  = meta_num    # [N, 3] float32
        self.meta_cat  = meta_cat    # [N, 2] int64

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        text  = str(row["text"])
        label = int(row["label"])

        # ── Load image ────────────────────────────────────────────────────────
        img_path = row.get("image_path")
        try:
            raw_img = (Image.open(img_path).convert("RGB")
                       if img_path else None)
        except Exception:
            raw_img = None
        clip_img = raw_img if raw_img is not None else _CLIP_PLACEHOLDER
        vilt_img = raw_img if raw_img is not None else _VILT_PLACEHOLDER

        # ── CLIP: image + text. The text tokens let CLIP contribute even when
        # Kaggle has no image folder and images become grey placeholders.
        clip_out = self.clip_proc(
            text=text,
            images=clip_img,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77,
        )
        clip_pixels = clip_out["pixel_values"].squeeze(0)   # [3, 224, 224]
        clip_input_ids = clip_out["input_ids"].squeeze(0)    # [77]
        clip_attention_mask = clip_out["attention_mask"].squeeze(0)

        # ── ViLT: text + image (ViLT is a joint VLP model)
        vilt_out = self.vilt_proc(
            text=text,
            images=vilt_img,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=40,
        )
        input_ids      = vilt_out["input_ids"].squeeze(0)        # [40]
        token_type_ids = vilt_out["token_type_ids"].squeeze(0)   # [40]
        attention_mask = vilt_out["attention_mask"].squeeze(0)   # [40]
        vilt_pixels    = vilt_out["pixel_values"].squeeze(0)     # [3,384,384]
        vilt_pmask     = vilt_out.get("pixel_mask")
        if vilt_pmask is not None:
            vilt_pmask = vilt_pmask.squeeze(0)

        item = dict(
            clip_pixel_values     = clip_pixels,
            clip_input_ids        = clip_input_ids,
            clip_attention_mask   = clip_attention_mask,
            input_ids             = input_ids,
            token_type_ids        = token_type_ids,
            attention_mask        = attention_mask,
            vilt_pixel_values     = vilt_pixels,
            meta_num              = torch.tensor(self.meta_num[idx], dtype=torch.float32),
            meta_cat              = torch.tensor(self.meta_cat[idx], dtype=torch.long),
            labels                = torch.tensor(label,              dtype=torch.long),
        )
        if vilt_pmask is not None:
            item["pixel_mask"] = vilt_pmask
        return item


def dual_collate(batch):
    """Stack all tensor fields; handles optional pixel_mask gracefully."""
    keys = batch[0].keys()
    return {k: torch.stack([b[k] for b in batch]) for k in keys}


print("FakedditDualDataset defined ✓")


FakedditDualDataset defined ✓


In [11]:
# ── Cell 6: Cross-Attention Fusion Model (CLIP × ViLT) ───────────────────────
#
# Data flow:
#
#  CLIP vision encoder (ViT-L/14)
#    pixel_values [B,3,224,224] → last_hidden_state [B, 257, 1024]
#    drop CLS → patch tokens    [B, 256, 1024]
#
#  ViLT encoder (vilt-b32-mlm)
#    input_ids + vilt_pixel_values → last_hidden_state [B, Nseq, 768]
#    slice text portion            → text tokens        [B,  40,  768]
#
#  Project to FUSION_DIM=256 each:
#    V_proj [B, 256, 256]
#    T_proj [B,  40, 256]
#
#  Stacked bidirectional cross-attention (FUSION_LAYERS=2):
#    Layer i:
#      V_new = CrossAttn(Q=V, KV=T)   ← vision attends text
#      T_new = CrossAttn(Q=T, KV=V)   ← text  attends vision
#
#  MeanPool(V_final) → [B,256]
#  MeanPool(T_final) → [B,256]
#  concat            → [B,512]
#
#  MetadataMLP       → [B, 64]
#
#  concat(512+64)=576 → LayerNorm → MLP → 2-class logits


# ── CrossAttentionLayer ───────────────────────────────────────────────────────
class CrossAttentionLayer(nn.Module):
    """
    Single cross-attention block:
        Q from one modality, K/V from the other.
    Residual + LayerNorm + position-wise FFN (pre-norm style).
    """
    def __init__(self, dim, num_heads, dropout=0.1):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.norm_q  = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn    = nn.MultiheadAttention(
            dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2   = nn.LayerNorm(dim)
        self.ffn     = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, query, key_value, key_padding_mask=None):
        # Pre-norm
        q  = self.norm_q(query)
        kv = self.norm_kv(key_value)
        attn_out, _ = self.attn(q, kv, kv,
                                key_padding_mask=key_padding_mask,
                                need_weights=False)
        x = query + self.drop(attn_out)          # residual
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x                                  # [B, Nq, dim]


# ── BidirectionalCrossAttentionFusion ────────────────────────────────────────
class BidirectionalCrossAttentionFusion(nn.Module):
    """
    Projects CLIP patch tokens (1024-d) and ViLT text tokens (768-d) to
    FUSION_DIM (256), then runs FUSION_LAYERS of bidirectional cross-attention.

    Output: [B, FUSION_DIM*2]  (mean-pooled vision ++ mean-pooled text)
    """
    def __init__(self,
                 vision_dim  = CLIP_VISION_DIM,   # 1024
                 text_dim    = VILT_TEXT_DIM,      # 768
                 fusion_dim  = FUSION_DIM,         # 256
                 num_heads   = FUSION_HEADS,       # 8
                 num_layers  = FUSION_LAYERS,      # 2
                 dropout     = 0.1):
        super().__init__()
        # Input projections: align both modalities to fusion_dim
        self.v_proj = nn.Sequential(
            nn.Linear(vision_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.Dropout(dropout),
        )
        self.t_proj = nn.Sequential(
            nn.Linear(text_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.Dropout(dropout),
        )
        self.clip_text_proj = nn.Sequential(
            nn.Linear(text_dim, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.Dropout(dropout),
        )
        # Stacked bidirectional cross-attention pairs
        self.v_layers = nn.ModuleList([
            CrossAttentionLayer(fusion_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.t_layers = nn.ModuleList([
            CrossAttentionLayer(fusion_dim, num_heads, dropout)
            for _ in range(num_layers)
        ])
        self.out_dim = fusion_dim * 3   # vision + ViLT text + CLIP text

    def forward(self, vision_tokens, text_tokens, clip_text_tokens=None, text_pad_mask=None, clip_text_pad_mask=None):
        """
        Args:
            vision_tokens : [B, N_v, 1024]  CLIP patch tokens
            text_tokens   : [B, N_t,  768]  ViLT word tokens
            text_pad_mask : [B, N_t]  bool  True = padding position
        Returns:
            fused : [B, 512]
        """
        v = self.v_proj(vision_tokens)   # [B, N_v, 256]
        t = self.t_proj(text_tokens)     # [B, N_t, 256]

        for v_layer, t_layer in zip(self.v_layers, self.t_layers):
            # Vision attends to text  (text can be padded → pass mask)
            v_new = v_layer(query=v, key_value=t, key_padding_mask=text_pad_mask)
            # Text attends to vision  (vision has no padding)
            t_new = t_layer(query=t, key_value=v)
            v, t = v_new, t_new

        v_pool = v.mean(dim=1)                         # [B, 256]
        t_pool = t.mean(dim=1)                         # [B, 256]
        if clip_text_tokens is not None:
            ct = self.clip_text_proj(clip_text_tokens)
            if clip_text_pad_mask is not None:
                keep = (~clip_text_pad_mask).float().unsqueeze(-1)
                ct_pool = (ct * keep).sum(dim=1) / keep.sum(dim=1).clamp_min(1.0)
            else:
                ct_pool = ct.mean(dim=1)
        else:
            ct_pool = torch.zeros_like(t_pool)
        return torch.cat([v_pool, t_pool, ct_pool], dim=1)  # [B, 768]


# ── MetadataMLP ───────────────────────────────────────────────────────────────
class MetadataMLP(nn.Module):
    """
    Encodes numeric (3) + categorical (2 × emb_dim) metadata → 64-d vector.
    """
    def __init__(self,
                 domain_vocab    = META_DOMAIN_VOCAB + 1,
                 subreddit_vocab = META_SUBREDDIT_VOCAB + 1,
                 emb_dim         = META_CAT_EMB_DIM,
                 hidden          = META_HIDDEN,
                 num_feat        = META_NUM_FEATURES,
                 dropout         = 0.2):
        super().__init__()
        self.domain_emb    = nn.Embedding(domain_vocab,    emb_dim, padding_idx=0)
        self.subreddit_emb = nn.Embedding(subreddit_vocab, emb_dim, padding_idx=0)
        in_dim = num_feat + 2 * emb_dim
        self.mlp = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout + 0.15),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
        )
        self.out_dim = hidden   # 64

    def forward(self, meta_num, meta_cat):
        # meta_num : [B, 3]   meta_cat : [B, 2]
        if not USE_METADATA:
            return meta_num.new_zeros((meta_num.size(0), self.out_dim))
        d = self.domain_emb(meta_cat[:, 0])      # [B, 16]
        s = self.subreddit_emb(meta_cat[:, 1])   # [B, 16]
        x = torch.cat([meta_num, d, s], dim=1)   # [B, 35]
        return self.mlp(x)                        # [B, 64]


# ── CLIPViLTClassifier ────────────────────────────────────────────────────────
class CLIPViLTClassifier(nn.Module):
    """
    End-to-end model:
      CLIP  →  patch token sequence        [B, 256, 1024]
      ViLT  →  text token sequence         [B,  40,  768]
      BidirectionalCrossAttentionFusion  → [B, 512]
      MetadataMLP                        → [B,  64]
      concat                             → [B, 576]
      classifier MLP                     → [B,   2]

    Freezing strategy (fits T4 16 GB / P100 16 GB):
      • CLIP vision : all frozen except last 2 ViT blocks
      • ViLT encoder: all frozen except last 2 transformer layers
      • Fusion + MetadataMLP + Classifier: fully trainable
    """
    def __init__(self, clip_model, vilt_model, meta_mlp,
                 num_classes=NUM_CLASSES, dropout=DROPOUT_RATE):
        super().__init__()
        self.clip     = clip_model
        self.vilt     = vilt_model
        self.meta_mlp = meta_mlp
        self.fusion   = BidirectionalCrossAttentionFusion()

        # ── Freeze CLIP; optionally unfreeze only the final blocks.
        for p in self.clip.parameters():
            p.requires_grad = False
        n_unfreeze = max(0, int(UNFREEZE_LAST_BLOCKS))
        if n_unfreeze > 0:
            vision_layers = self.clip.vision_model.encoder.layers
            for layer in vision_layers[-n_unfreeze:]:
                for p in layer.parameters():
                    p.requires_grad = True
            text_layers = self.clip.text_model.encoder.layers
            for layer in text_layers[-n_unfreeze:]:
                for p in layer.parameters():
                    p.requires_grad = True
            for p in self.clip.vision_model.post_layernorm.parameters():
                p.requires_grad = True
            for p in self.clip.text_model.final_layer_norm.parameters():
                p.requires_grad = True

        # ── Freeze ViLT; optionally unfreeze only the final layers.
        for p in self.vilt.parameters():
            p.requires_grad = False
        if n_unfreeze > 0:
            for layer in self.vilt.encoder.layer[-n_unfreeze:]:
                for p in layer.parameters():
                    p.requires_grad = True

        # ── Classification head
        head_in = self.fusion.out_dim + meta_mlp.out_dim
        self.classifier = nn.Sequential(
            nn.LayerNorm(head_in),
            nn.Linear(head_in, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 192),
            nn.GELU(),
            nn.Dropout(dropout * 0.75),
            nn.Linear(192, num_classes),
        )

    def forward(self,
                clip_pixel_values,   # [B, 3, 224, 224]
                clip_input_ids,      # [B, 77]
                clip_attention_mask, # [B, 77]
                input_ids,           # [B, 40]
                token_type_ids,      # [B, 40]
                attention_mask,      # [B, 40]
                vilt_pixel_values,   # [B, 3, 384, 384]
                meta_num,            # [B, 3]
                meta_cat,            # [B, 2]
                pixel_mask=None):    # [B, H, W] optional ViLT pixel mask

        # ── CLIP: extract patch token sequence ───────────────────────────────
        # vision_model output: last_hidden_state [B, 257, 1024]  (CLS + 256 patches)
        clip_vision_out = self.clip.vision_model(
            pixel_values=clip_pixel_values,
            output_hidden_states=False,
            return_dict=True,
        )
        # Drop CLS token (index 0) → pure patch tokens
        vision_tokens = clip_vision_out.last_hidden_state[:, 1:, :]  # [B, 256, 1024]

        clip_text_out = self.clip.text_model(
            input_ids=clip_input_ids,
            attention_mask=clip_attention_mask,
            output_hidden_states=False,
            return_dict=True,
        )
        clip_text_tokens = clip_text_out.last_hidden_state  # [B, 77, 768]
        clip_text_pad_mask = (clip_attention_mask == 0)

        # ── ViLT: extract text token sequence ────────────────────────────────
        # ViLT concatenates text & image patches internally; last_hidden_state
        # shape: [B, text_seq + image_patches + specials, 768]
        # Text tokens occupy the first `input_ids.shape[1]` positions.
        vilt_kwargs = dict(
            input_ids      = input_ids,
            token_type_ids = token_type_ids,
            attention_mask = attention_mask,
            pixel_values   = vilt_pixel_values,
        )
        if pixel_mask is not None:
            vilt_kwargs["pixel_mask"] = pixel_mask
        vilt_out    = self.vilt(**vilt_kwargs, return_dict=True)
        n_text      = input_ids.shape[1]
        text_tokens = vilt_out.last_hidden_state[:, :n_text, :]  # [B, 40, 768]

        # Padding mask for cross-attention: True = ignore (padding position)
        text_pad_mask = (attention_mask == 0)    # [B, 40]

        # ── Bidirectional cross-attention fusion ──────────────────────────────
        fused = self.fusion(
            vision_tokens,
            text_tokens,
            clip_text_tokens=clip_text_tokens,
            text_pad_mask=text_pad_mask,
            clip_text_pad_mask=clip_text_pad_mask,
        )

        # ── Metadata branch ───────────────────────────────────────────────────
        meta  = self.meta_mlp(meta_num, meta_cat)                       # [B,  64]

        # ── Final classification ──────────────────────────────────────────────
        return self.classifier(torch.cat([fused, meta], dim=1))         # [B,   2]


def count_trainable(model):
    total    = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen   = total - trainable
    print(f"  Parameters — total: {total:,}  |  trainable: {trainable:,}  "
          f"({100*trainable/total:.1f}%)  |  frozen: {frozen:,}")


print("CLIPViLTClassifier defined ✓")


CLIPViLTClassifier defined ✓


In [12]:
# ── Cell 7: Training Helpers ──────────────────────────────────────────────────

USE_AMP = (DEVICE == "cuda")


class EarlyStopping:
    def __init__(self, patience=PATIENCE, delta=1e-4):
        self.patience   = patience
        self.delta      = delta
        self.best_score = None
        self.counter    = 0
        self.best_state = None
        self.stop       = False

    def step(self, score, model):
        # Higher is better. The caller passes validation F1 penalised by the
        # train/validation accuracy gap, so overfit checkpoints are not selected.
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score = score
            self.best_state = {k: v.cpu().clone()
                               for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore_best(self, model):
        if self.best_state:
            model.load_state_dict(
                {k: v.to(DEVICE) for k, v in self.best_state.items()})


def _to_device(batch):
    """Pop labels & metadata; move everything to DEVICE."""
    labels   = batch.pop("labels").to(DEVICE)
    meta_num = batch.pop("meta_num").to(DEVICE)
    meta_cat = batch.pop("meta_cat").to(DEVICE)
    batch    = {k: v.to(DEVICE) for k, v in batch.items()}
    return batch, labels, meta_num, meta_cat


def _forward(model, batch):
    """Forward pass — maps batch dict keys to model.forward signature."""
    b, labels, meta_num, meta_cat = _to_device(batch)
    logits = model(
        clip_pixel_values   = b["clip_pixel_values"],
        clip_input_ids      = b["clip_input_ids"],
        clip_attention_mask = b["clip_attention_mask"],
        input_ids           = b["input_ids"],
        token_type_ids      = b["token_type_ids"],
        attention_mask      = b["attention_mask"],
        vilt_pixel_values = b["vilt_pixel_values"],
        meta_num          = meta_num,
        meta_cat          = meta_cat,
        pixel_mask        = b.get("pixel_mask"),
    )
    return logits, labels


def train_epoch(model, loader, optimizer, criterion, scheduler, scaler):
    model.train()
    total_loss = correct = n = 0
    nb_steps = len(loader)

    for step, batch in enumerate(loader, 1):
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits, labels = _forward(model, batch)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += labels.size(0)

        if step % 20 == 0 or step == nb_steps:
            print(f"    step {step:3d}/{nb_steps} | "
                  f"loss={total_loss/n:.4f} | acc={correct/n:.4f}", flush=True)

    scheduler.step()
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    preds, trues = [], []

    for batch in loader:
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits, labels = _forward(model, batch)
            loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds.extend(logits.argmax(1).cpu().numpy())
        trues.extend(labels.cpu().numpy())

    n = len(trues)
    return dict(
        loss      = total_loss / n,
        accuracy  = accuracy_score(trues, preds),
        f1        = f1_score(trues, preds, average="weighted", zero_division=0),
        precision = precision_score(trues, preds, average="weighted", zero_division=0),
        recall    = recall_score(trues, preds, average="weighted", zero_division=0),
    ), np.array(preds), np.array(trues)


def leakage_warning(train_acc, val_acc, val_f1):
    return train_acc >= PERFECT_SCORE_WARNING and val_acc >= PERFECT_SCORE_WARNING and val_f1 >= PERFECT_SCORE_WARNING


print("Training helpers ready ✓")


Training helpers ready ✓


In [13]:
# ── Cell 8: 10-Fold Cross-Validation Loop ─────────────────────────────────────

def run_kfold(df, meta_fitter):
    """
    10-fold stratified CV with CLIP × ViLT cross-attention fusion + controlled metadata.

    Both base models are loaded once on CPU, weights are reset each fold to
    ensure fair evaluation without cross-fold leakage.
    """
    labels  = df["label"].values
    indices = np.arange(len(df))
    groups  = df["group_key"].values if "group_key" in df.columns else indices

    if USE_GROUP_SPLIT:
        skf = StratifiedGroupKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
        split_iter = skf.split(indices, labels, groups)
        print("Using StratifiedGroupKFold: duplicate/similar titles stay in the same fold.")
    else:
        skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
        split_iter = skf.split(indices, labels)

    fold_results, all_history, all_cms = [], [], []

    # ── Initial GPU status ────────────────────────────────────────────────────
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print(f"GPU memory before model load: {free/1e9:.1f}/{total/1e9:.1f} GB free")

    # ── Load both base models to CPU once ─────────────────────────────────────
    print(f"\nLoading CLIP ({CLIP_MODEL}) ...")
    clip_proc  = CLIPProcessor.from_pretrained(CLIP_MODEL)
    clip_base  = CLIPModel.from_pretrained(CLIP_MODEL).cpu()
    clip_state = {k: v.cpu().clone() for k, v in clip_base.state_dict().items()}
    print(f"  ✓ CLIP loaded  ({sum(p.numel() for p in clip_base.parameters())/1e6:.0f}M params)")

    print(f"\nLoading ViLT ({VILT_MODEL}) ...")
    vilt_proc  = ViltProcessor.from_pretrained(VILT_MODEL)
    vilt_base  = ViltModel.from_pretrained(VILT_MODEL).cpu()
    vilt_state = {k: v.cpu().clone() for k, v in vilt_base.state_dict().items()}
    print(f"  ✓ ViLT loaded  ({sum(p.numel() for p in vilt_base.parameters())/1e6:.0f}M params)")

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    for fold, (train_val_idx, test_idx) in enumerate(split_iter, 1):
        print(f"\n{'='*70}")
        print(f"  FOLD {fold} / {NUM_FOLDS}")
        print(f"{'='*70}")

        # ── 80 / 10 / 10 split ───────────────────────────────────────────────
        # With 10 folds, each outer fold is the test set. Validation is a
        # stratified slice from the remaining training folds, so the test fold
        # stays untouched until final evaluation.
        train_idx, val_idx = train_test_split(
            train_val_idx,
            test_size=1.0 / (NUM_FOLDS - 1),
            random_state=SEED + fold,
            stratify=labels[train_val_idx],
        )
        test_idx_f = test_idx
        print(f"  Train:{len(train_idx)}  Val:{len(val_idx)}  Test:{len(test_idx_f)}")

        # Fit metadata on TRAIN ONLY. Fitting on all rows leaks validation/test distribution.
        fold_meta = MetadataEncoderFitter().fit(df.iloc[train_idx])
        train_meta_num, train_meta_cat = fold_meta.transform(df.iloc[train_idx])
        val_meta_num, val_meta_cat = fold_meta.transform(df.iloc[val_idx])
        test_meta_num, test_meta_cat = fold_meta.transform(df.iloc[test_idx_f])

        # ── Build datasets & loaders ──────────────────────────────────────────
        def make_ds(idx, meta_num, meta_cat):
            return FakedditDualDataset(
                df.iloc[idx], clip_proc, vilt_proc,
                meta_num, meta_cat,
            )

        nw  = 2 if DEVICE == "cuda" else 0
        pin = DEVICE == "cuda"
        train_loader = DataLoader(make_ds(train_idx, train_meta_num, train_meta_cat), BATCH_SIZE, shuffle=True,
                                  collate_fn=dual_collate, num_workers=nw, pin_memory=pin)
        val_loader   = DataLoader(make_ds(val_idx, val_meta_num, val_meta_cat), BATCH_SIZE, shuffle=False,
                                  collate_fn=dual_collate, num_workers=nw, pin_memory=pin)
        test_loader  = DataLoader(make_ds(test_idx_f, test_meta_num, test_meta_cat), BATCH_SIZE, shuffle=False,
                                  collate_fn=dual_collate, num_workers=nw, pin_memory=pin)

        # ── Reset weights and move to GPU ─────────────────────────────────────
        print("  Initialising model for this fold ...")
        clip_base.load_state_dict(clip_state, strict=True)
        vilt_base.load_state_dict(vilt_state, strict=True)
        clip_base.to(DEVICE)
        vilt_base.to(DEVICE)

        if DEVICE == "cuda":
            free, total = torch.cuda.mem_get_info()
            print(f"  GPU after backbone load: {free/1e9:.1f}/{total/1e9:.1f} GB free")

        meta_mlp = MetadataMLP(
            domain_vocab    = fold_meta.domain_vocab_size,
            subreddit_vocab = fold_meta.subreddit_vocab_size,
        ).to(DEVICE)

        model = CLIPViLTClassifier(clip_base, vilt_base, meta_mlp).to(DEVICE)
        count_trainable(model)

        criterion  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
        backbone_params, head_params = [], []
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if name.startswith("clip.") or name.startswith("vilt."):
                backbone_params.append(param)
            else:
                head_params.append(param)
        optimizer  = AdamW(
            [
                {"params": head_params, "lr": LEARNING_RATE},
                {"params": backbone_params, "lr": LR_BACKBONE},
            ],
            weight_decay=WEIGHT_DECAY,
        )
        scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=5e-7)
        early_stop = EarlyStopping(PATIENCE)
        history    = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[],
                      "val_f1":[], "overfit_gap":[], "monitor_score":[]}

        # ── Epoch loop ────────────────────────────────────────────────────────
        for epoch in range(1, EPOCHS + 1):
            print(f"\n  ── Epoch {epoch}/{EPOCHS} ──────────────────────────────────")
            tr_loss, tr_acc = train_epoch(
                model, train_loader, optimizer, criterion, scheduler, scaler)
            val_m, _, _ = evaluate(model, val_loader, criterion)

            gap = max(0.0, tr_acc - val_m["accuracy"])
            monitor_score = val_m["f1"] - OVERFIT_GAP_PENALTY * gap
            if leakage_warning(tr_acc, val_m["accuracy"], val_m["f1"]):
                monitor_score -= 0.05
                print("  WARNING: near-perfect train/validation metrics detected; treating as possible shortcut leakage.")

            history["train_loss"].append(tr_loss)
            history["val_loss"].append(val_m["loss"])
            history["train_acc"].append(tr_acc)
            history["val_acc"].append(val_m["accuracy"])
            history["val_f1"].append(val_m["f1"])
            history["overfit_gap"].append(gap)
            history["monitor_score"].append(monitor_score)
            early_stop.step(monitor_score, model)

            marker = "  ← best" if early_stop.counter == 0 else ""
            print(f"  E{epoch:2d}/{EPOCHS} | "
                  f"TrL={tr_loss:.4f} TrA={tr_acc:.4f} | "
                  f"VaL={val_m['loss']:.4f} VaA={val_m['accuracy']:.4f} "
                  f"VaF1={val_m['f1']:.4f} Gap={gap:.4f} Score={monitor_score:.4f}{marker}")

            if early_stop.stop:
                print(f"  ⏹  Early stopping triggered at epoch {epoch}.")
                break

        # ── Evaluate on test set (best checkpoint) ────────────────────────────
        early_stop.restore_best(model)
        test_m, preds, trues = evaluate(model, test_loader, criterion)
        cm = confusion_matrix(trues, preds)
        all_cms.append(cm)

        print(f"\n  ── Fold {fold} Test Results ─────────────────────────────────")
        print(f"  Accuracy  : {test_m['accuracy']:.4f}")
        print(f"  F1-Score  : {test_m['f1']:.4f}")
        print(f"  Precision : {test_m['precision']:.4f}")
        print(f"  Recall    : {test_m['recall']:.4f}")
        print()
        print(classification_report(trues, preds, target_names=["Real", "Fake"]))

        best_epoch = int(np.argmax(history["monitor_score"]) + 1)
        fold_results.append({
            "Fold"        : fold,
            "Accuracy"    : round(test_m["accuracy"],  4),
            "F1-Score"    : round(test_m["f1"],        4),
            "Precision"   : round(test_m["precision"], 4),
            "Recall"      : round(test_m["recall"],    4),
            "Best Epoch"  : best_epoch,
            "Overfit Gap" : round(history["overfit_gap"][best_epoch - 1], 4),
        })
        all_history.append(history)

        # ── Free GPU memory before next fold ──────────────────────────────────
        clip_base.to("cpu")
        vilt_base.to("cpu")
        del model, meta_mlp, optimizer, scheduler, early_stop
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            free, total = torch.cuda.mem_get_info()
            print(f"  GPU after cleanup: {free/1e9:.1f}/{total/1e9:.1f} GB free")

    return fold_results, all_history, all_cms


print("K-fold runner ready ✓")


K-fold runner ready ✓


In [14]:
# ── Cell 9: Results & Visualisation ──────────────────────────────────────────

def print_results_table(fold_results):
    print("\n" + "="*70)
    print("  📊 CLIP × ViLT RESULTS  —  2-WAY CLASSIFICATION")
    print("="*70)
    rows  = [[r["Fold"], r["Accuracy"], r["F1-Score"], r["Precision"], r["Recall"],
              r.get("Best Epoch", "-"), r.get("Overfit Gap", "-")]
             for r in fold_results]
    accs  = [r["Accuracy"]  for r in fold_results]
    f1s   = [r["F1-Score"]  for r in fold_results]
    precs = [r["Precision"] for r in fold_results]
    recs  = [r["Recall"]    for r in fold_results]
    gaps  = [r.get("Overfit Gap", 0.0) for r in fold_results]
    rows.append([
        "Mean ± Std",
        f"{np.mean(accs):.4f} ± {np.std(accs):.4f}",
        f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}",
        f"{np.mean(precs):.4f} ± {np.std(precs):.4f}",
        f"{np.mean(recs):.4f} ± {np.std(recs):.4f}",
        "-",
        f"{np.mean(gaps):.4f} ± {np.std(gaps):.4f}",
    ])
    print(tabulate(rows,
                   headers=["Fold","Accuracy","F1-Score","Precision","Recall","Best Epoch","Overfit Gap"],
                   tablefmt="fancy_grid"))
    print("="*70)
    return dict(
        mean_accuracy  = np.mean(accs),
        mean_f1        = np.mean(f1s),
        mean_precision = np.mean(precs),
        mean_recall    = np.mean(recs),
    )


def save_results_csv(fold_results):
    dfr  = pd.DataFrame(fold_results)
    path = f"{OUTPUT_DIR}/clip_vilt_results.csv"
    dfr.to_csv(path, index=False)
    print(f"✓ Results saved: {path}")
    try:    display(dfr)
    except: print(dfr.to_string())


def plot_visualizations(all_history, all_cms):
    C = ["#2196F3", "#FF5722"]

    # Training curves
    fig, axes = plt.subplots(2, NUM_FOLDS, figsize=(5*NUM_FOLDS, 8))
    for i, h in enumerate(all_history):
        ep = range(1, len(h["train_loss"]) + 1)
        ax = axes[0, i]
        ax.plot(ep, h["train_loss"], color=C[0], lw=2, label="Train")
        ax.plot(ep, h["val_loss"],   color=C[1], lw=2, ls="--", label="Val")
        ax.set_title(f"Fold {i+1} — Loss", fontweight="bold")
        ax.legend(); ax.grid(alpha=0.3)

        ax = axes[1, i]
        ax.plot(ep, h["train_acc"], color=C[0], lw=2, label="Train")
        ax.plot(ep, h["val_acc"],   color=C[1], lw=2, ls="--", label="Val")
        ax.set_title(f"Fold {i+1} — Accuracy", fontweight="bold")
        ax.set_ylim(0, 1); ax.legend(); ax.grid(alpha=0.3)

    plt.suptitle("CLIP × ViLT Training Curves  (2-way Fake News Detection)",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/training_curves.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {path}")

    # Confusion matrices
    fig, axes = plt.subplots(1, NUM_FOLDS, figsize=(5*NUM_FOLDS, 4))
    for i, cm in enumerate(all_cms):
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i],
                    xticklabels=["Real","Fake"],
                    yticklabels=["Real","Fake"])
        axes[i].set_title(f"Fold {i+1}", fontweight="bold")
        axes[i].set_xlabel("Predicted")
        axes[i].set_ylabel("True")
    plt.suptitle("CLIP × ViLT Confusion Matrices",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/confusion_matrices.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {path}")


print("Visualisation helpers ready ✓")


Visualisation helpers ready ✓


In [15]:
# ── Cell 10: Load Fakeddit + Fit Metadata Encoder ─────────────────────────────
print("Loading Fakeddit dataset  (2-way + metadata)...\n")
df_main = load_fakeddit(NUM_SAMPLES)

# Metadata encoders are fitted inside each fold on TRAIN ONLY.
# This prevents validation/test distribution leakage.
meta_fitter = None

print("\nSample rows:")
cols = ["text","label","score","num_comments","upvote_ratio","domain","subreddit", "group_key"]
try:    display(df_main[cols].head(3))
except: print(df_main[cols].head(3).to_string())


Loading Fakeddit dataset  (2-way + metadata)...

Scanning /kaggle/input for Fakeddit dataset...

  selected metadata: /kaggle/input/datasets/vanshikavmittal/fakeddit-dataset/multimodal_only_samples/multimodal_train.tsv
  Columns (first 15): ['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id', 'image_url', 'linked_submission_id', 'num_comments', 'score', 'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label']
  Numeric meta   : ['score', 'num_comments', 'upvote_ratio']
  Categorical    : ['domain', 'subreddit']
  Duplicate title groups removed: 59684
  ⚠️  No image directory found — text+metadata only mode.

✓ Balanced: 12000 samples  (Real=6000 | Fake=6000)
  With images: 0

Sample rows:


,text,label,score,num_comments,upvote_ratio,domain,subreddit,group_key
0,after naias bad luck feng shui expert eyed,1,5,0.0,0.72,philstar.com,nottheonion,after naias bad luck feng shui expert eyed
1,well the movie just got a little bit more inte...,0,64,0.0,0.00,nan,psbattle_artwork,well the movie just got a little bit more inte...
2,reposting my tiny chicken pie now with bread a...,1,10,9.0,1.00,i.redd.it,mildlyinteresting,reposting my tiny chicken pie now with bread a...


In [16]:
# ── Cell 11: Run 10-Fold Cross-Validation ─────────────────────────────────────
# Estimated time on Kaggle T4: longer than 5-fold CV because NUM_FOLDS=10.
# Speed tips: reduce NUM_SAMPLES, EPOCHS, or use P100/A100 accelerator.
print(f"\nStarting {NUM_FOLDS}-fold CV  [CLIP × ViLT cross-attention + controlled metadata]...\n")
fold_results, all_history, all_cms = run_kfold(df_main, meta_fitter)



Starting 10-fold CV  [CLIP × ViLT cross-attention + controlled metadata]...

Using StratifiedGroupKFold: duplicate/similar titles stay in the same fold.
GPU memory before model load: 15.5/15.6 GB free

Loading CLIP (openai/clip-vit-large-patch14) ...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

ViltModel LOAD REPORT from: dandelin/vilt-b32-mlm
Key                                          | Status     |  | 
---------------------------------------------+------------+--+-
vilt.embeddings.text_embeddings.position_ids | UNEXPECTED |  | 
mlm_score.bias                               | UNEXPECTED |  | 
mlm_score.transform.dense.bias               | UNEXPECTED |  | 
mlm_score.decoder.weight                     | UNEXPECTED |  | 
mlm_score.transform.dense.weight             | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.bias           | UNEXPECTED |  | 
mlm_score.transform.LayerNorm.weight         | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

  ✓ ViLT loaded  (112M params)

  FOLD 1 / 10
  Train:9600  Val:1200  Test:1200
  [domain] vocab size = 101  (incl. <unk>)
  [subreddit] vocab size = 1  (incl. <unk>)
Metadata encoder fitted ✓
  Initialising model for this fold ...
  GPU after backbone load: 13.3/15.6 GB free
  Parameters — total: 543,543,481  |  trainable: 31,107,512  (5.7%)  |  frozen: 512,435,969

  ── Epoch 1/10 ──────────────────────────────────
    step  20/1200 | loss=0.6774 | acc=0.5437
    step  40/1200 | loss=0.6828 | acc=0.5437
    step  60/1200 | loss=0.6770 | acc=0.5625
    step  80/1200 | loss=0.6718 | acc=0.5828
    step 100/1200 | loss=0.6604 | acc=0.6038
    step 120/1200 | loss=0.6577 | acc=0.6104
    step 140/1200 | loss=0.6451 | acc=0.6277
    step 160/1200 | loss=0.6378 | acc=0.6391
    step 180/1200 | loss=0.6268 | acc=0.6507
    step 200/1200 | loss=0.6237 | acc=0.6562
    step 220/1200 | loss=0.6148 | acc=0.6642
    step 240/1200 | loss=0.6099 | acc=0.6708
    step 260/1200 | loss=0.6049 | acc=0

KeyboardInterrupt: 

In [ ]:
# ── Cell 12: Display Results ──────────────────────────────────────────────────
summary = print_results_table(fold_results)
save_results_csv(fold_results)


In [ ]:
# ── Cell 13: Generate Visualisations ──────────────────────────────────────────
plot_visualizations(all_history, all_cms)


In [ ]:
# ── Cell 14: Final Summary ────────────────────────────────────────────────────
print("\n" + "="*70)
print("  🎯  FINAL SUMMARY")
print("="*70)
print(f"  Vision backbone : CLIP ViT-L/14  ({CLIP_MODEL})")
print(f"  Text   backbone : ViLT-B/32      ({VILT_MODEL})")
print(f"  Fusion          : Bidirectional Cross-Attention")
print(f"                    dim={FUSION_DIM}  heads={FUSION_HEADS}  layers={FUSION_LAYERS}")
print(f"  Metadata        : score · num_comments · upvote_ratio · domain · subreddit")
print(f"  Task            : 2-way (Real vs Fake)  —  3-way & 6-way DISABLED")
print(f"  Dataset         : Fakeddit  ({NUM_SAMPLES} balanced samples, {NUM_FOLDS}-fold CV)")
print()
print(f"  📊 Cross-validated Performance:")
print(f"     Accuracy  : {summary['mean_accuracy']:.4f}")
print(f"     F1-Score  : {summary['mean_f1']:.4f}")
print(f"     Precision : {summary['mean_precision']:.4f}")
print(f"     Recall    : {summary['mean_recall']:.4f}")
print()
print(f"  💾 Output files saved to  {OUTPUT_DIR}/")
print(f"     clip_vilt_results.csv")
print(f"     training_curves.png")
print(f"     confusion_matrices.png")
print("="*70)
print("  ✅  Training complete!")
print("="*70)
